## 1. 파일 읽기

In [184]:
import glob
import pandas as pd

# 배민 파일 31개를 하나의 DataFrame으로 합치기
baemin_files = sorted(glob.glob("data/baemin/baemin*.csv"))

baemin_list = []

for file in baemin_files:
    df_temp = pd.read_csv(file)
    baemin_list.append(df_temp)

baemin_df = pd.concat(baemin_list, ignore_index=True)


# 쿠팡 파일 31개를 하나의 DataFrame으로 합치기
coupang_files = sorted(glob.glob("data/coupang/coupang*.csv"))

coupang_list = []

for file in coupang_files:
    df_temp = pd.read_csv(file)
    coupang_list.append(df_temp)

coupang_df = pd.concat(coupang_list, ignore_index=True)


# 결과 확인
print("===== 배민 DataFrame =====")
display(baemin_df)

print("===== 쿠팡 DataFrame =====")
display(coupang_df)

===== 배민 DataFrame =====


,order_at,total_bill,ad_yn,cpc
0,2026-10-01 11:48:34,26000,Y,700
1,2026-10-01 11:53:22,15000,N,0
2,2026-10-01 12:18:52,35000,N,0
3,2026-10-01 12:28:44,35000,N,0
4,2026-10-01 12:50:50,35000,N,0
...,...,...,...,...
1150,2026-10-31 20:34:53,17000,N,0
1151,2026-10-31 20:47:07,38000,Y,640
1152,2026-10-31 21:39:35,19000,Y,690
1153,2026-10-31 21:45:29,45000,N,0


===== 쿠팡 DataFrame =====


,order_time,total_price,advertisement_yn,ad_pct
0,2026-10-01 11:46:07,18000,Y,10
1,2026-10-01 12:06:50,21000,Y,10
2,2026-10-01 12:16:29,58000,Y,10
3,2026-10-01 12:19:49,43000,Y,10
4,2026-10-01 13:00:05,58000,N,0
...,...,...,...,...
707,2026-10-31 19:33:36,32000,N,0
708,2026-10-31 19:36:08,18000,Y,5
709,2026-10-31 19:50:22,18000,N,0
710,2026-10-31 19:59:04,34000,N,0


## 2. 칼럼명 통일

In [185]:
# 두 플랫폼의 데이터프레임 칼럼명 변경
baemin_df = baemin_df.rename(
    columns={
        "order_at": "주문일시",
        "total_bill": "전체금액",
        "ad_yn": "광고유입여부",
    }
)

coupang_df = coupang_df.rename(
    columns={
        "order_time": "주문일시",
        "pay_amt": "전체금액",
        "advertisement_yn": "광고유입여부",
    }
)

# 플랫폼 칼럼 추가
baemin_df["플랫폼"] = "배민"
coupang_df["플랫폼"] = "쿠팡"

# 광고집행비 칼럼 추가
baemin_df["광고집행비"] = baemin_df["cpc"]

coupang_df["광고집행비"] = (
    coupang_df["total_price"] * coupang_df["ad_pct"] / 100
)

# 결과 확인
print("===== 배민 DataFrame =====")
display(baemin_df)

print("===== 쿠팡 DataFrame =====")
display(coupang_df)

===== 배민 DataFrame =====


,주문일시,전체금액,광고유입여부,cpc,플랫폼,광고집행비
0,2026-10-01 11:48:34,26000,Y,700,배민,700
1,2026-10-01 11:53:22,15000,N,0,배민,0
2,2026-10-01 12:18:52,35000,N,0,배민,0
3,2026-10-01 12:28:44,35000,N,0,배민,0
4,2026-10-01 12:50:50,35000,N,0,배민,0
...,...,...,...,...,...,...
1150,2026-10-31 20:34:53,17000,N,0,배민,0
1151,2026-10-31 20:47:07,38000,Y,640,배민,640
1152,2026-10-31 21:39:35,19000,Y,690,배민,690
1153,2026-10-31 21:45:29,45000,N,0,배민,0


===== 쿠팡 DataFrame =====


,주문일시,total_price,광고유입여부,ad_pct,플랫폼,광고집행비
0,2026-10-01 11:46:07,18000,Y,10,쿠팡,1800.0
1,2026-10-01 12:06:50,21000,Y,10,쿠팡,2100.0
2,2026-10-01 12:16:29,58000,Y,10,쿠팡,5800.0
3,2026-10-01 12:19:49,43000,Y,10,쿠팡,4300.0
4,2026-10-01 13:00:05,58000,N,0,쿠팡,0.0
...,...,...,...,...,...,...
707,2026-10-31 19:33:36,32000,N,0,쿠팡,0.0
708,2026-10-31 19:36:08,18000,Y,5,쿠팡,900.0
709,2026-10-31 19:50:22,18000,N,0,쿠팡,0.0
710,2026-10-31 19:59:04,34000,N,0,쿠팡,0.0


## 3. 데이터 통합

In [186]:
# 두 DataFrame 하나로 합치기
total_df = pd.concat([baemin_df, coupang_df], ignore_index=True)

# '주문일시'를 datetime 형식으로 변환하고 날짜만 담은 '주문일자' 칼럼 추가
total_df["주문일시"] = pd.to_datetime(total_df["주문일시"])
total_df["주문일자"] = total_df["주문일시"].dt.date

# 주문일시 순으로 정렬
total_df = total_df.sort_values(by="주문일시", ascending=True).reset_index(drop=True)

# 결과 확인
print("===== 배달어플 통합 Dataframe =====")
display(total_df)

===== 배달어플 통합 Dataframe =====


,주문일시,전체금액,광고유입여부,cpc,플랫폼,광고집행비,total_price,ad_pct,주문일자
0,2026-10-01 11:46:07,NaN,Y,NaN,쿠팡,1800.0,18000.0,10.0,2026-10-01
1,2026-10-01 11:48:34,26000.0,Y,700.0,배민,700.0,NaN,NaN,2026-10-01
2,2026-10-01 11:53:22,15000.0,N,0.0,배민,0.0,NaN,NaN,2026-10-01
3,2026-10-01 12:06:50,NaN,Y,NaN,쿠팡,2100.0,21000.0,10.0,2026-10-01
4,2026-10-01 12:16:29,NaN,Y,NaN,쿠팡,5800.0,58000.0,10.0,2026-10-01
...,...,...,...,...,...,...,...,...,...
1862,2026-10-31 20:47:07,38000.0,Y,640.0,배민,640.0,NaN,NaN,2026-10-31
1863,2026-10-31 20:49:23,NaN,N,NaN,쿠팡,0.0,24000.0,0.0,2026-10-31
1864,2026-10-31 21:39:35,19000.0,Y,690.0,배민,690.0,NaN,NaN,2026-10-31
1865,2026-10-31 21:45:29,45000.0,N,0.0,배민,0.0,NaN,NaN,2026-10-31


## 4. 일자별 매출 집계

In [187]:
# 행(주문일자), 열(플랫폼), 값(전체금액 합계)으로 표 생성
pivot_df = total_df.pivot_table(
    index="주문일자", columns="플랫폼", values="전체금액", aggfunc="sum"
)

# '합계' 칼럼 추가
pivot_df["합계"] = pivot_df.sum(axis=1)

# 결과 확인
print("===== 매출 정리 =====")
display(pivot_df)

===== 매출 정리 =====


플랫폼,배민,쿠팡,합계
주문일자,,,
2026-10-01,959000.0,0.0,959000.0
2026-10-02,1348000.0,0.0,1348000.0
2026-10-03,1294000.0,0.0,1294000.0
2026-10-04,1127000.0,0.0,1127000.0
2026-10-05,1103000.0,0.0,1103000.0
2026-10-06,1028000.0,0.0,1028000.0
2026-10-07,1227000.0,0.0,1227000.0
2026-10-08,946000.0,0.0,946000.0
2026-10-09,1200000.0,0.0,1200000.0


## 5. 광고 효과 분석

In [188]:
# 플랫폼별 광고유입 주문의 비율
mask = total_df["광고유입여부"] == "Y"
ad_counts = total_df[mask].groupby("플랫폼")["광고유입여부"].count()

total_counts = total_df.groupby("플랫폼")["광고유입여부"].count()

ad_ratio = (ad_counts / total_counts) * 100

# 플랫폼별 광고집행비 합계
ad_sum = total_df.groupby("플랫폼")["광고집행비"].sum()

#결과 확인
print("===== 플랫폼별 광고 유입 주문 비율(%) =====")
print(ad_ratio)

print("\n===== 플랫폼별 광고집행비 합계 =====")
print(ad_sum)

===== 플랫폼별 광고 유입 주문 비율(%) =====
플랫폼
배민    34.978355
쿠팡    51.966292
Name: 광고유입여부, dtype: float64

===== 플랫폼별 광고집행비 합계 =====
플랫폼
배민    243170.0
쿠팡    886700.0
Name: 광고집행비, dtype: float64


## 6. 식자재비 추출

In [189]:
# 은행 거래내역에서 거래처가 식자재마트인 출금 내역
bank_df = pd.read_csv("data/bank_20261001_20261031.csv", encoding="euc-kr")

mart_df = bank_df[bank_df["거래처"] == "식자재마트"]
print("===== 식자재마트 출금내역 =====")
display(mart_df)

===== 식자재마트 출금내역 =====


,거래일시,적요,거래처,출금액,입금액,잔액
1,2026.10.01 09:01,체크카드,식자재마트,197100,0,4796800
5,2026.10.02 09:15,체크카드,식자재마트,267100,0,6193900
7,2026.10.02 15:03,체크카드,식자재마트,113300,0,6056000
10,2026.10.03 09:07,체크카드,식자재마트,194800,0,5437000
11,2026.10.03 15:28,체크카드,식자재마트,52800,0,5384200
13,2026.10.04 09:08,체크카드,식자재마트,289300,0,5082700
15,2026.10.05 09:44,체크카드,식자재마트,214400,0,4859900
20,2026.10.06 09:04,체크카드,식자재마트,209500,0,6969700
24,2026.10.07 09:03,체크카드,식자재마트,273100,0,6540300
26,2026.10.07 15:16,체크카드,식자재마트,59700,0,6470700


In [190]:
#일자별 식자재비 합계
mart_df["거래일자"] = pd.to_datetime(mart_df["거래일시"]).dt.date
daily_mart_cost = mart_df.groupby("거래일자")["출금액"].sum()
print("\n===== 일자별 식자재비 합계 ======")
display(daily_mart_cost)


===== 일자별 식자재비 합계 ======


거래일자
2026-10-01    197100
2026-10-02    380400
2026-10-03    247600
2026-10-04    289300
2026-10-05    214400
2026-10-06    209500
2026-10-07    332800
2026-10-08    219300
2026-10-09    351300
2026-10-10    324900
2026-10-11    225000
2026-10-12    305500
2026-10-13    314100
2026-10-14    375200
2026-10-15    288500
2026-10-16    389700
2026-10-17    342000
2026-10-18    184600
2026-10-19    250100
2026-10-20    205600
2026-10-21    292400
2026-10-22    263500
2026-10-23    289100
2026-10-24    289800
2026-10-25    312300
2026-10-26    272800
2026-10-27    200400
2026-10-28    391000
2026-10-29    213300
2026-10-30    302800
2026-10-31    397300
Name: 출금액, dtype: int64

## 7. 최종 결과표 저장

In [191]:
# 최종 결과표
pivot_df["식자재비"] = daily_mart_cost
pivot_df["매출-식자재비"] = pivot_df["합계"] - pivot_df["식자재비"]

final_df = pivot_df.reset_index()

# 결과 확인
display(final_df)

# 결과 저장
final_df.to_csv("result.csv", index=False, encoding="utf-8-sig")

플랫폼,주문일자,배민,쿠팡,합계,식자재비,매출-식자재비
0,2026-10-01,959000.0,0.0,959000.0,197100,761900.0
1,2026-10-02,1348000.0,0.0,1348000.0,380400,967600.0
2,2026-10-03,1294000.0,0.0,1294000.0,247600,1046400.0
3,2026-10-04,1127000.0,0.0,1127000.0,289300,837700.0
4,2026-10-05,1103000.0,0.0,1103000.0,214400,888600.0
5,2026-10-06,1028000.0,0.0,1028000.0,209500,818500.0
6,2026-10-07,1227000.0,0.0,1227000.0,332800,894200.0
7,2026-10-08,946000.0,0.0,946000.0,219300,726700.0
8,2026-10-09,1200000.0,0.0,1200000.0,351300,848700.0
9,2026-10-10,1379000.0,0.0,1379000.0,324900,1054100.0
